<!-- Cache bust 12_decorators_notebook -->

# Decorators

***

### 🔹 1. First-Class Functions & Closures Recap
To fully understand decorators, we must remember two key concepts in Python:
1. **First-Class Functions:** Functions in Python are objects. They can be assigned to variables, passed as arguments to other functions, and returned from functions.
2. **Closures:** A nested function that retains access to variables from its outer enclosing scope even after the outer function has finished executing.

In [ ]:
# 1. Passing functions as arguments
def loud(text):
    return text.upper()

def quiet(text):
    return text.lower()

def greet(func):
    # 'func' is a variable pointing to a function object
    greeting = func("Hello, World!")
    print(greeting)

greet(loud)
greet(quiet)

In [ ]:
# 2. Closure Example
def parent_function(person_name):
    # Enclosing scope variable
    def child_function():
        return f"Greetings, {person_name}!"
    return child_function # Returns the function object without executing it

greet_kamran = parent_function("Kamran")
print("Closure reference:", greet_kamran)
print("Result of calling closure:", greet_kamran())

***

### 🔹 2. What is a Decorator?
A **Decorator** is a design pattern in Python that allows you to extend or modify the behavior of a callable (function or class) without permanently modifying its code.

* Under the hood, a decorator is a function that takes another function as an argument, wraps its behavior in an inner function, and returns that inner function.
* Python provides the `@decorator_name` syntactic sugar to make wrapping functions highly readable.

In [ ]:
# Defining a basic decorator
def my_decorator(func):
    def wrapper():
        print("[Log] Something is happening before the function is called.")
        func() # Execute the original function
        print("[Log] Something is happening after the function is called.")
    return wrapper

# Using syntactic sugar to apply the decorator
@my_decorator
def say_hello():
    print("Hello!")

# Call the decorated function
say_hello()

***

### 🔹 3. Decorators for Functions with Arguments
If you try to decorate a function that takes arguments using the decorator above, it will raise a `TypeError` because the inner `wrapper()` function accepts no arguments.
* **Solution:** Use `*args` and `**kwargs` in the inner wrapper function to accept any inputs and pass them directly to the original function.

In [ ]:
def log_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"Calling function: '{func.__name__}' with positional args: {args} and keyword args: {kwargs}")
        result = func(*args, **kwargs)
        print(f"Function '{func.__name__}' finished execution.")
        return result
    return wrapper

@log_decorator
def add_nums(x, y):
    return x + y

result = add_nums(10, 25)
print("Final Sum Result:", result)

***

### 🔹 4. Decorators Accepting Arguments
Sometimes we need to pass parameters to the decorator itself (e.g. `@repeat(3)`). This requires **three nested functions**:
1. The outermost function accepts the decorator parameters.
2. The middle function receives the target function to decorate.
3. The innermost function handles the wrapping logic (`*args` and `**kwargs`).

In [ ]:
def repeat(num_times):
    def decorator_repeat(func):
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(num_times=3)
def greet_client(name):
    print(f"Hello, {name}!")

greet_client("John")

***

### 🔹 5. Chaining Multiple Decorators
You can apply multiple decorators to a single function.
* **Order of Execution:** Decorators are applied **from bottom to top (inside-out)**.
* In mathematical terms, `@dec1` followed by `@dec2` on `func` is equivalent to `dec1(dec2(func))`.

In [ ]:
def make_bold(func):
    def wrapper(*args, **kwargs):
        return "<b>" + func(*args, **kwargs) + "</b>"
    return wrapper

def make_italic(func):
    def wrapper(*args, **kwargs):
        return "<i>" + func(*args, **kwargs) + "</i>"
    return wrapper

@make_bold
@make_italic
def format_text(txt):
    return txt

print(format_text("Python Rules"))

***

### 🔹 6. Preserving Function Metadata (`@functools.wraps`)
* When a function is decorated, its identity is replaced by the inner `wrapper` function.
* This means metadata like `__name__` and `__doc__` (docstrings) are lost and replaced by the wrapper's metadata.
* **Solution:** Import **`functools.wraps`** and decorate the inner `wrapper` function with it to preserve the original function's identity.

In [ ]:
import functools

def without_wraps(func):
    def wrapper(*args, **kwargs):
        """This is the wrapper docstring."""
        return func(*args, **kwargs)
    return wrapper

def with_wraps(func):
    @functools.wraps(func) # Preserves identity!
    def wrapper(*args, **kwargs):
        """This is the wrapper docstring."""
        return func(*args, **kwargs)
    return wrapper

@without_wraps
def dummy_one():
    """This is the original docstring."""
    pass

@with_wraps
def dummy_two():
    """This is the original docstring."""
    pass

print("Without wraps name:", dummy_one.__name__)
print("Without wraps doc:", dummy_one.__doc__)
print("-" * 30)
print("With wraps name:", dummy_two.__name__)
print("With wraps doc:", dummy_two.__doc__)

***

### 🔹 7. Real-world Decorator Patterns (AI/ML & Engineering)

#### 🔹 1. Execution Timer (Latency Analysis)
Measures the elapsed time taken to run models or operations.

In [ ]:
import time

def execution_timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        elapsed = end_time - start_time
        print(f"[Timer] Function '{func.__name__}' took {elapsed:.6f} seconds to execute.")
        return result
    return wrapper

@execution_timer
def run_heavy_loop():
    total = sum(i * i for i in range(10000000))
    return total

run_heavy_loop()

#### 🔹 2. Memoization / Caching (Performance speedup)
Saves results of expensive function computations to avoid redundant calls.

In [ ]:
from functools import lru_cache

# Using Python's built-in Least Recently Used cache decorator
@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

start = time.perf_counter()
print("40th Fibonacci (with cache):", fibonacci(40))
end = time.perf_counter()
print(f"Time taken: {end - start:.6f} seconds")

***

### 🔹 8. Class-Based Decorators
We can write decorators as classes by implementing the **`__call__`** dunder method. This is useful for state-preserving decorators (e.g. counting how many times a function has been called).

In [ ]:
class CallCounter:
    def __init__(self, func):
        self.func = func
        self.count = 0
        
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"[Counter] Function '{self.func.__name__}' has been called {self.count} times.")
        return self.func(*args, **kwargs)

@CallCounter
def process_data(data):
    return data * 2

print(process_data(5))
print(process_data(10))
print(process_data(20))

***

## 📝 Practice Questions


### 🟢 Easy Level


In [ ]:
#Q1 Demonstrate functions as first-class citizens by passing a function `square` as argument to `apply`.

In [ ]:
#Q2 Define a nested inner function inside an outer function and execute it.

In [ ]:
#Q3 Write a function that returns another function.

In [ ]:
#Q4 Write a basic decorator function `logger` that prints 'Before' and 'After' execution.

In [ ]:
#Q5 Write a decorator that logs arguments and return value of any function.

In [ ]:
#Q6 Write a decorator to convert string outputs of functions to uppercase.

In [ ]:
#Q7 Demonstrate stacking two decorators on a single function.

In [ ]:
#Q8 Write a decorator to count and print how many times a function has been executed.

In [ ]:
#Q9 Write a decorator that validates if arguments passed are non-empty strings.

In [ ]:
#Q10 Write a decorator that prints a custom header line before calling the decorated function.

### 🟡 Medium Level


In [ ]:
#Q11 Show that wrapping functions manually removes original metadata, and fix it using `functools.wraps`.

In [ ]:
#Q12 Create a parameterized decorator `repeat(num)` that executes the wrapped function `num` times.

In [ ]:
#Q13 Write a timing decorator that prints the execution time of functions in milliseconds.

In [ ]:
#Q14 Write a slow-down decorator that inserts a delay of 0.1 seconds before executing the function.

In [ ]:
#Q15 Write a type check decorator that checks if input argument types match the specified annotations.

In [ ]:
#Q16 Write a decorator that logs function calls and arguments to a text file.

In [ ]:
#Q17 Write a decorator that handles exceptions by returning a default value instead of crashing.

In [ ]:
#Q18 Write a class decorator that turns a class definition into a Singleton pattern.

In [ ]:
#Q19 Implement a memoization decorator to optimize calculation speed of recursive Fibonacci.

In [ ]:
#Q20 Write a class-based decorator using the magic method `__call__`.

In [ ]:
#Q21 Write a class decorator that adds a class attribute `timestamp` with the current time.

In [ ]:
#Q22 Write a deprecation decorator that warns the user when calling a deprecated function.

### 🔴 Hard Level


In [ ]:
#Q23 Implement a rate-limiting decorator that restricts calls of a function (e.g. max 2 calls per second).

In [ ]:
#Q24 Create a parameterized class-based decorator that records execution history.

In [ ]:
#Q25 Implement an authorization checking decorator matching user roles (e.g. admin role requirement).

In [ ]:
#Q26 Write a decorator that validates outputs of prediction functions (checks float is between 0 and 1).

In [ ]:
#Q27 Implement a dynamic caching decorator with a custom cache size limit (evicts oldest item).

In [ ]:
#Q28 Write a decorator that prints memory usage optimization options when functions return large lists.

In [ ]:
#Q29 Implement a function overload decorator matching behavior dynamically based on argument count.

In [ ]:
#Q30 Write a decorator that executes a function asynchronously in a background thread.